# Lesson 17 — Conic Optimization

## Learning objectives

1. State the standard form of LP, SOCP, SDP and the chain of inclusions.
2. Recognize problems naturally formulated as SOCP (norm constraints, robust LP, portfolio).
3. Recognize SDP applications (LMIs, max-cut, polynomial optimization).
4. Solve SOCP-representable models in `discopt`, and recognize geometric programs, which are convex after a log transform (`discopt.gp`).

## 1. Cones

A cone $K \subseteq \mathbb{R}^n$ is closed under non-negative scaling. The relevant cones for optimization {cite:p}`BenTal2001,Boyd2004`:

- **Linear cone** $\mathbb{R}^n_{\ge 0}$ — the LP cone.
- **Lorentz cone** (second-order cone) $\{(t, x) : t \ge \|x\|_2\}$.
- **PSD cone** $\mathbb{S}_+^n = \{X = X^\top, X \succeq 0\}$.

A *conic program*:

$$\min c^\top x \;\;\text{s.t.}\;\; Ax = b, \; x \in K.$$

LP, SOCP, SDP are conic programs over their respective cones. All three are convex; all three are polynomial-time via interior-point methods.

## 2. SOCP

$$\min c^\top x \;\;\text{s.t.}\;\; \|A_i x + b_i\|_2 \le c_i^\top x + d_i, \; i = 1, \dots, m.$$

Common applications {cite:p}`BenTal2001`:

- **Robust LP** with ellipsoidal uncertainty becomes an SOCP (Lesson 25).
- **Markowitz portfolio**: minimize $\|L^\top w\|_2$ where $\Sigma = LL^\top$.
- **Truss design**, antenna design, geometric medians.

QP and convex QCQP are special cases of SOCP.

## 3. SDP

$$\min \langle C, X \rangle \;\;\text{s.t.}\;\; \langle A_i, X \rangle = b_i, \; X \succeq 0.$$

Generalizes both LP (diagonal $X$) and SOCP (block structure). Applications:

- **Max-cut relaxation** (Goemans-Williamson): cut value $\le$ SDP value $\le$ trivial UB.
- **LMI control problems**: stability, $H_\infty$ control.
- **Polynomial optimization**: via sum-of-squares (Lasserre hierarchy).

SDPs are polynomial-time, but constants are large; solvers (Mosek, SDPT3, SeDuMi, COSMO) are slower than LP solvers.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model, sqrt
# (For sum-of-squares write `dm.sum(x**2)` — there is no separate `sum_squares` in discopt.)

# Markowitz: min sigma^2  s.t.  mu' w >= r_min, sum(w)=1, w>=0
n = 6
rng = np.random.default_rng(0)
mu = 0.05 + 0.1 * rng.random(n)              # expected returns
L  = rng.normal(scale=0.05, size=(n, n))     # covariance factor
Sigma = L @ L.T + 1e-3 * np.eye(n)

m = Model("portfolio")
w = m.continuous("w", shape=(n,), lb=0)
m.subject_to(dm.sum(w) == 1)
m.subject_to(mu @ w >= 0.08)
m.minimize(w @ Sigma @ w)                    # convex QP / SOCP
r = m.solve()
print("portfolio:", r.value(w).round(3), "  variance:", r.objective)

# The same model in explicit SOCP form: min t  s.t. ||L^T w||_2 <= t.
# discopt has no dedicated conic (SOCP/SDP) solver. Its convexity detector
# proves this model convex, and a single interior-point NLP solve (POUNCE)
# certifies the global optimum (r.convex_fast_path == True, no branching).
Lc = np.linalg.cholesky(Sigma)
ms = Model("portfolio_socp")
ws = ms.continuous("w", shape=(n,), lb=0)
t = ms.continuous("t", lb=0, ub=10)
ms.subject_to(dm.sum(ws) == 1)
ms.subject_to(mu @ ws >= 0.08)
ms.subject_to(dm.norm(Lc.T @ ws) <= t)
ms.minimize(t)
rs = ms.solve()
print(f"SOCP form: t^2 = {rs.objective**2:.6f}  convex fast path: {rs.convex_fast_path}")


## 4. The conic chain

$$\text{LP} \subset \text{QP} \subset \text{SOCP} \subset \text{SDP}.$$

Choosing the *smallest* cone that fits your problem is best — solvers exploit structure. A QP solved as an SDP is far slower than as an SOCP {cite:p}`BenTal2001`.

## 5. Strong duality and slater

Conic duality is weaker than LP duality. Strong duality (zero gap, dual optimum attained) holds under a constraint qualification such as Slater's condition: strict feasibility, meaning a point in the *interior* of the cone. Without it, the optimal value may not be attained, or a nonzero duality gap may appear, which cannot happen for feasible LPs {cite:p}`BenTal2001`. Interior-point conic solvers then struggle, and need special handling such as facial reduction or homogeneous self-dual embeddings.

## 6. Geometric programming: convex in disguise

A **monomial** is $c\,x_1^{a_1}\cdots x_n^{a_n}$ with $c > 0$, $a_i \in \mathbb{R}$, $x > 0$; a **posynomial** is a sum of monomials. A *geometric program* minimizes a posynomial subject to posynomial $\le 1$ and monomial $= 1$ constraints {cite:p}`Boyd2007gp`. In $x$ it is not convex. Under $x_i = e^{u_i}$, though, every monomial becomes $\exp(a^\top u + \log c)$, so posynomial constraints become log-sum-exp constraints (convex) and monomial equalities become linear {cite:p}`Duffin1967`. The exponential cone is the conic home of these problems. Sizing and design problems (circuits, structures, aspect ratios) fit this template.

`discopt.gp` recognizes this structure in an ordinary model (`classify_gp`), builds the log-space convex model (`as_geometric_program`), and solves it (`solve_gp`). Because the transformed problem is convex, an `optimal` status is a *global* certificate with no branch-and-bound. Variables must be continuous with a strictly positive lower bound.

In [ ]:
from discopt.gp import classify_gp, solve_gp

# min x/y + 2xy  s.t.  xy + x/y <= 10,  1/x <= 1 (i.e. x >= 1).
# Closed form: x* = 1, y* = 1/sqrt(2), f* = 2*sqrt(2).
g = Model("gp_demo")
x = g.continuous("x", lb=1e-4, ub=1e4)
y = g.continuous("y", lb=1e-4, ub=1e4)
g.minimize(x / y + 2.0 * x * y)
g.subject_to(x * y + x / y <= 10.0)
g.subject_to(1.0 / x <= 1.0)

print("recognized as a GP:", classify_gp(g) is not None)
rg = solve_gp(g)
print(f"status = {rg.status}, f* = {float(rg.objective):.6f} (2*sqrt(2) = {2*np.sqrt(2):.6f})")
print({k: round(float(v), 6) for k, v in rg.x.items()})

## References
{cite:p}`BenTal2001,Boyd2004` are the textbooks; {cite:p}`Wright1997` for IPM theory; {cite:p}`Boyd2007gp,Duffin1967` for geometric programming.